## Methodologies for Adapting Local Large Language Models for Reliable Transformation of Unstructured Clinical Notes into Structured EHR Data
**Master thesis by Mohammad Omar Sharieff** (target platform: OpenEMR, canonical model: HL7 FHIR R4)

**Notebook 01: data preparation.** Loads the raw MTSamples export, cleans it, draws a seeded random sample, and splits it into a development set (may be inspected freely) and a test set (touched once, at the very end). The notes are *not* gold standard yet; they only become gold standard after manual annotation against the target schema.

In [13]:
import hashlib, json, platform
from pathlib import Path
import pandas as pd

SEED = 42
RAW_PATH = Path("data/raw/mtsamples.csv")      # raw data: never modify
OUT_DIR = Path("data/processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Specialties kept (note: "Consult - History and Phy." is NOT necessarily outpatient)
SPECIALTIES = ["Consult - History and Phy.", "General Medicine"]
N_DEV, N_TEST = 40, 40                          # adjust to what you can realistically annotate

### Confirmation of each original row having unique ID:

In [14]:
assert df["source_id"].notna().all(), "Some source IDs are missing."
assert df["source_id"].is_unique, "Some source IDs are repeated."

### 1. Load raw data (with provenance)

In [15]:
raw_sha256 = hashlib.sha256(RAW_PATH.read_bytes()).hexdigest()
print("Raw file SHA-256:", raw_sha256)

df = pd.read_csv(RAW_PATH, index_col=0)
df.index.name = "source_id"          # keep the original row id as a stable identifier
df = df.reset_index()

print(f"Total notes loaded: {len(df)}")
print("Columns:", df.columns.tolist())

Raw file SHA-256: 83a65fd7ca071930264e27e6fb011bc5303a21294ff734d7a8198cfbb1db7a2a
Total notes loaded: 4999
Columns: ['source_id', 'description', 'medical_specialty', 'sample_name', 'transcription', 'keywords']


### 2. Clean *before* counting or filtering

In [16]:
n0 = len(df)

# Remove extra spaces at the beginning and end.
df["medical_specialty"] = df["medical_specialty"].str.strip()
df["transcription"] = df["transcription"].str.strip()

# Remove rows where the note is missing or empty.
df = df.dropna(subset=["transcription"])
df = df[df["transcription"] != ""].copy()
n1 = len(df)

# Do not remove duplicate notes here.
# We will do that AFTER selecting the specialties.

print(f"Loaded {n0} notes → {n1} non-empty notes")
print("\nTop 10 specialties:")
print(df["medical_specialty"].value_counts().head(10))

Loaded 4999 notes → 4966 non-empty notes

Top 10 specialties:
medical_specialty
Surgery                          1088
Consult - History and Phy.        516
Cardiovascular / Pulmonary        371
Orthopedic                        355
Radiology                         273
General Medicine                  259
Gastroenterology                  224
Neurology                         223
SOAP / Chart / Progress Notes     166
Urology                           156
Name: count, dtype: int64


### 3. Filter to encounter-type notes

In [17]:
# FIRST: keep only the specialties we want.
filtered = df[df["medical_specialty"].isin(SPECIALTIES)].copy()
n_selected = len(filtered)

# THEN: remove repeated notes within these selected specialties.
filtered = filtered.drop_duplicates(subset=["transcription"]).copy()
n2 = len(filtered)

# Give each remaining note an ID and calculate its length.
filtered["note_id"] = filtered["source_id"].map(
    lambda i: f"mts_{i:04d}"
)
filtered["n_chars"] = filtered["transcription"].str.len()

print(f"Notes in selected specialties: {n_selected}")
print(f"Repeated notes removed: {n_selected - n2}")
print(f"Unique notes remaining: {n2}")

print("\nSpecialty counts:")
print(filtered["medical_specialty"].value_counts())

print("\nNote length (characters):")
print(filtered["n_chars"].describe())

Notes in selected specialties: 775
Repeated notes removed: 158
Unique notes remaining: 617

Specialty counts:
medical_specialty
Consult - History and Phy.    358
General Medicine              259
Name: count, dtype: int64

Note length (characters):
count      617.000000
mean      3520.766613
std       2019.477597
min         12.000000
25%       2162.000000
50%       3145.000000
75%       4294.000000
max      15216.000000
Name: n_chars, dtype: float64


### Checking if enough notes remain

In [18]:
assert len(filtered) >= N_DEV + N_TEST, (
    f"Need {N_DEV + N_TEST} notes, but only {len(filtered)} remain."
)

### 4. Seeded random sample, then dev/test split
The split is decided here, once, before any annotation or prompt work. The test file must not be opened until the final evaluation.

In [19]:
sample = filtered.sample(n=N_DEV + N_TEST, random_state=SEED).reset_index(drop=True)
dev = sample.iloc[:N_DEV].copy()
test = sample.iloc[N_DEV:].copy()

assert set(dev["note_id"]).isdisjoint(test["note_id"])
assert set(dev["transcription"]).isdisjoint(test["transcription"])

print("Dev specialty mix:\n", dev["medical_specialty"].value_counts(), sep="")
print("\nTest specialty mix:\n", test["medical_specialty"].value_counts(), sep="")

Dev specialty mix:
medical_specialty
General Medicine              20
Consult - History and Phy.    20
Name: count, dtype: int64

Test specialty mix:
medical_specialty
Consult - History and Phy.    25
General Medicine              15
Name: count, dtype: int64


### 5. Save (CSV/JSONL for notes, no Excel)
Excel cells are capped at 32,767 characters and silently mangle long text, so notes are stored as CSV/JSONL. Excel/spreadsheet is only worth it later for an annotation sheet.

In [20]:
cols = ["note_id", "source_id", "medical_specialty", "sample_name", "description", "keywords", "transcription"]

filtered[cols].to_csv(OUT_DIR / "all_encounters.csv", index=False)
dev[cols].to_json(OUT_DIR / "dev_notes.jsonl", orient="records", lines=True, force_ascii=False)
test[cols].to_json(OUT_DIR / "test_notes.jsonl", orient="records", lines=True, force_ascii=False)

manifest = {
    "seed": SEED,
    "raw_file": str(RAW_PATH),
    "raw_sha256": raw_sha256,
    "specialties": SPECIALTIES,
    "counts": {
        "loaded": n0,
        "non_empty": n1,
        "selected_before_deduplication": n_selected,
        "filtered": len(filtered),
        "dev": len(dev),
        "test": len(test),
    },
    "dev_ids": dev["note_id"].tolist(),
    "test_ids": test["note_id"].tolist(),
    "python": platform.python_version(),
    "pandas": pd.__version__,
}
(OUT_DIR / "split_manifest.json").write_text(json.dumps(manifest, indent=2))
print("Saved dev/test notes and split_manifest.json to", OUT_DIR)

Saved dev/test notes and split_manifest.json to data\processed
